# FT-NCFM Colab A100 runner

This notebook runs the auditable MNIST Mini-VLA proxy on a hosted Colab GPU. It is an infrastructure and optimization-convergence experiment, not a reproduction of the paper's robotics success rates.

Run the setup, test, preflight, and benchmark cells first. The longer 5% cells are disabled by default and must be enabled deliberately after reviewing the benchmark timing. Completed seeds are copied to Google Drive and skipped when the same run is resumed.

In [ ]:
# Reproducible, secret-free settings.
REPO_URL = "https://github.com/hppddub/NCFM.git"
UPSTREAM_URL = "https://github.com/gszfwsb/NCFM.git"
BRANCH = "codex/ft-ncfm-replication"
PINNED_SHA = ""  # Optional: paste a previously recorded manifest SHA to rerun it exactly.
REPO_DIR = "/content/NCFM"
DRIVE_ROOT = "/content/drive/MyDrive/FT-NCFM"
PIN_PAPER_TORCH = True
REQUIRE_A100 = True
print({"branch": BRANCH, "pinned_sha": PINNED_SHA or "current branch head", "drive_root": DRIVE_ROOT})

In [ ]:
# Hardware gate. This deliberately uses nvidia-smi before importing or replacing PyTorch.
import csv
import io
import subprocess

query = subprocess.check_output([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader,nounits",
], text=True).strip()
name, memory_mib, driver = next(csv.reader(io.StringIO(query), skipinitialspace=True))
print({"gpu": name, "memory_mib": int(memory_mib), "driver": driver})
if REQUIRE_A100 and "A100" not in name.upper():
    raise RuntimeError(f"Expected an A100, but Colab allocated {name}. Change the runtime and reconnect.")

In [ ]:
# Persist completed seeds outside the disposable Colab VM.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or fast-forward the dedicated replication branch. No credentials are required.
import os
from pathlib import Path

repo = Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", BRANCH], check=True)
remotes = subprocess.check_output(["git", "-C", str(repo), "remote"], text=True).split()
if "upstream" not in remotes:
    subprocess.run(["git", "-C", str(repo), "remote", "add", "upstream", UPSTREAM_URL], check=True)
subprocess.run(["git", "-C", str(repo), "fetch", "--no-tags", "upstream", "main"], check=True)
if PINNED_SHA:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", PINNED_SHA], check=True)
    subprocess.run(["git", "-C", str(repo), "switch", "--detach", PINNED_SHA], check=True)
git_sha = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
os.chdir(repo)
print({"repository": str(repo), "git_sha": git_sha})

In [ ]:
# Install the paper's PyTorch/CUDA family, then the reproduction package.
import importlib.metadata
import sys

def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

def base_version(version):
    return version.split("+")[0] if version else None

before = {name: installed_version(name) for name in ("torch", "torchvision", "numpy")}
print("Before installation:", before)
if PIN_PAPER_TORCH and (base_version(before["torch"]) != "2.5.0" or base_version(before["torchvision"]) != "0.20.0"):
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "torch==2.5.0", "torchvision==0.20.0",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "numpy==2.1.3", "PyYAML==6.0.2",
    "pytest>=8.3", "ruff>=0.8",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(repo)], check=True)

import torch
import torchvision
assert torch.cuda.is_available(), "PyTorch cannot see the Colab GPU"
if PIN_PAPER_TORCH:
    assert torch.__version__.split("+")[0] == "2.5.0"
    assert torchvision.__version__.split("+")[0] == "0.20.0"
print({"torch": torch.__version__, "torchvision": torchvision.__version__, "cuda": torch.version.cuda, "device": torch.cuda.get_device_name(0)})

In [ ]:
# CPU-safe component and infrastructure checks.
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=repo, check=True)

In [ ]:
# Persist the A100, CUDA, package, Colab, and Git preflight without starting an experiment.
runner = [
    sys.executable, "infra/colab/run_experiment.py",
    "--config", "configs/ft_ncfm/minivla_nested_5pct_smoke.yaml",
    "--persistent-root", DRIVE_ROOT,
    "--run-name", "a100-preflight",
    "--require-gpu", "--require-a100", "--preflight-only",
]
subprocess.run(runner, cwd=repo, check=True)

In [ ]:
# Cheap timing gate: 1,000 source samples, seed 42.
RUN_1000_SAMPLE_BENCHMARK = True
if RUN_1000_SAMPLE_BENCHMARK:
    subprocess.run([
        sys.executable, "infra/colab/run_experiment.py",
        "--config", "configs/ft_ncfm/minivla_5pct.yaml",
        "--persistent-root", DRIVE_ROOT,
        "--run-name", "benchmark-1000",
        "--seeds", "42", "--max-samples", "1000",
        "--require-gpu", "--require-a100",
    ], cwd=repo, check=True)

In [ ]:
# Nested-fraction plumbing smoke: 3,000 source samples -> 150 feature proxies.
RUN_NESTED_SMOKE = True
if RUN_NESTED_SMOKE:
    subprocess.run([
        sys.executable, "infra/colab/run_experiment.py",
        "--config", "configs/ft_ncfm/minivla_nested_5pct_smoke.yaml",
        "--persistent-root", DRIVE_ROOT,
        "--run-name", "nested-smoke",
        "--seeds", "42",
        "--require-gpu", "--require-a100",
    ], cwd=repo, check=True)

In [ ]:
# True 5%-of-corpus proxy. Enable only after reviewing the 1,000-sample timing.
RUN_TRUE_5PCT_SEED_42 = False
if RUN_TRUE_5PCT_SEED_42:
    subprocess.run([
        sys.executable, "infra/colab/run_experiment.py",
        "--config", "configs/ft_ncfm/minivla_5pct.yaml",
        "--persistent-root", DRIVE_ROOT,
        "--run-name", "true-5pct",
        "--seeds", "42",
        "--require-gpu", "--require-a100",
    ], cwd=repo, check=True)

In [ ]:
# Remaining declared seeds. Enable only if seed 42 completes and its artifacts pass review.
RUN_REMAINING_TRUE_5PCT_SEEDS = False
if RUN_REMAINING_TRUE_5PCT_SEEDS:
    subprocess.run([
        sys.executable, "infra/colab/run_experiment.py",
        "--config", "configs/ft_ncfm/minivla_5pct.yaml",
        "--persistent-root", DRIVE_ROOT,
        "--run-name", "true-5pct",
        "--seeds", "123", "1024",
        "--require-gpu", "--require-a100",
    ], cwd=repo, check=True)

In [ ]:
# Compact handoff: display whichever aggregate results currently exist.
import json
for run_name in ("benchmark-1000", "nested-smoke", "true-5pct"):
    aggregate = Path(DRIVE_ROOT) / run_name / "aggregate_summary.json"
    if aggregate.is_file():
        print(f"\n=== {run_name} ===")
        print(json.dumps(json.loads(aggregate.read_text()), indent=2))